# Week 02 - Lesson 01 - Retrieval Augmented Generation (RAG)

This notebook introduces **Retrieval Augmented Generation (RAG)** - that combines information retrieval with generative AI to create intelligent systems that can access and reason over large knowledge bases. You'll learn how to build production-ready RAG systems using LangChain and LangGraph.

### Learning Objectives
1. Understand RAG architecture and its enterprise applications
2. Master document processing and vector storage techniques
3. Implement semantic search and retrieval systems
4. Build conversational RAG applications with memory

<div></div>
<img src="https://developer-blogs.nvidia.com/wp-content/uploads/2023/11/GenAI-Promo-AWS-DevNews-PRESS-1920x1080-1.png" alt="RAG System Diagram" style="max-width: 100%;" />

RAG systems are transforming enterprise knowledge management by enabling:

- **Intelligent Document Search**: Semantic search across corporate knowledge bases
- **Customer Support Automation**: Context-aware responses from product documentation
- **Internal Knowledge Assistants**: Employee self-service with company policies and procedures
- **Research and Development**: Accelerated information discovery and synthesis
- **Compliance and Audit**: Automated document analysis and regulatory compliance

Understanding RAG implementation is crucial for:

- Building enterprise knowledge management systems
- Creating intelligent customer support solutions
- Developing internal AI assistants
- Implementing document intelligence workflows
- Scaling AI applications with external knowledge sources

**In this lesson, before we dive deep into Agentic RAG, we should first make sure we understand what RAG in its vanilla form means.**

In [4]:
from IPython.display import IFrame

IFrame("https://drive.google.com/file/d/1WRAkDI0Cpyih8QSvZ-KViznSK-KplAY-/preview", 
       width=950, 
       height=300)


## 1. Environment Setup and Dependencies

First, let's ensure we have all necessary dependencies installed and configured for our RAG system.


In [1]:
# Install required dependencies for RAG system
%pip install -U --quiet langchain langchain-community langchain-openai langgraph chromadb

Note: you may need to restart the kernel to use updated packages.


In [ ]:
# Configure OpenAI API Key
OPENAI_API_KEY = "sk-your-open-ai-api-key"

In [3]:
# Import necessary libraries for RAG system
import os

# LangChain core imports
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.documents import Document

# Document processing imports
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Vector store imports
from langchain_community.vectorstores import Chroma

print("✅ All RAG dependencies imported successfully!")

✅ All RAG dependencies imported successfully!


## 2. Understanding RAG Architecture

Before implementing our RAG system, let's understand the fundamental architecture and some considerations:

### RAG System Components

1. **Document Ingestion**: Loading and processing various document formats
2. **Text Chunking**: Breaking documents into manageable pieces for processing
3. **Vector Embeddings**: Converting text chunks into numerical representations
4. **Vector Storage**: Storing embeddings in a searchable vector database
5. **Retrieval**: Finding relevant documents based on user queries
6. **Generation**: Using retrieved context to generate informed responses

## 3. Setting Up the Foundation

In [4]:
# Initialize the language model for generation
llm = ChatOpenAI(
    model="gpt-4o-mini",
    api_key=OPENAI_API_KEY,
    temperature=0.1,  # Lower temperature for more consistent, factual responses
    max_tokens=1000
)

# Initialize the embedding model for vector generation
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small",  # Cost-effective embedding model
    api_key=OPENAI_API_KEY
)

print("✅ Core models initialized successfully!")
print(f"LLM Model: {llm.model_name}")
print(f"Embedding Model: {embeddings.model}")
print(f"LLM Temperature: {llm.temperature}")
print(f"Max Tokens: {llm.max_tokens}")


✅ Core models initialized successfully!
LLM Model: gpt-4o-mini
Embedding Model: text-embedding-3-small
LLM Temperature: 0.1
Max Tokens: 1000


## 4. Document Processing and Chunking

RAG systems need to handle diverse document types efficiently. Let's implement a document processing pipeline:


In [5]:
# Create sample enterprise documents for demonstration
# In production, these would be loaded from your document management system

sample_documents = [
    Document(
        page_content="""
        Enterprise AI Strategy 2024
        
        Our company's AI strategy focuses on three key pillars:
        1. Customer Experience Enhancement through intelligent automation
        2. Operational Efficiency via process optimization
        3. Data-Driven Decision Making with advanced analytics
        
        Key initiatives include implementing RAG systems for customer support,
        deploying AI agents for internal knowledge management, and establishing
        data governance frameworks for AI model training.
        
        Budget allocation: $2.5M for AI infrastructure, $1.8M for talent acquisition,
        and $1.2M for compliance and security measures.
        """,
        metadata={"source": "enterprise_ai_strategy_2024.pdf", "department": "Strategy", "date": "2024-01-15"}
    ),
    Document(
        page_content="""
        Customer Support Best Practices
        
        Our customer support team follows these key principles:
        - Respond to all inquiries within 2 hours during business hours
        - Escalate complex technical issues to senior engineers
        - Maintain detailed logs of all customer interactions
        - Follow up on resolved issues within 48 hours
        
        Common support categories:
        1. Technical Issues (40 percent of tickets)
        2. Billing Questions (25 percent of tickets)
        3. Feature Requests (20 percent of tickets)
        4. Account Management (15 percent of tickets)
        
        Success metrics: 95 percent first-call resolution rate, 4.8/5 customer satisfaction score.
        """,
        metadata={"source": "customer_support_handbook.pdf", "department": "Support", "date": "2024-02-01"}
    ),
    Document(
        page_content="""
        Data Privacy and Security Policy
        
        All employees must comply with the following data handling requirements:
        
        Data Classification:
        - Public: Information that can be freely shared
        - Internal: Company information for employees only
        - Confidential: Sensitive business information
        - Restricted: Highly sensitive data requiring special handling
        
        Security Requirements:
        - All data must be encrypted in transit and at rest
        - Access controls must follow principle of least privilege
        - Regular security audits and penetration testing required
        - Incident response plan must be activated within 1 hour of breach detection
        
        Compliance: GDPR, CCPA, SOX, and industry-specific regulations apply.
        """,
        metadata={"source": "data_privacy_policy.pdf", "department": "Legal", "date": "2024-01-30"}
    )
]

print("✅ Sample documents created!")
print(f"Total documents: {len(sample_documents)}")
print("\nDocument sources:")
for i, doc in enumerate(sample_documents, 1):
    print(f"{i}. {doc.metadata['source']} - {doc.metadata['department']}")

✅ Sample documents created!
Total documents: 3

Document sources:
1. enterprise_ai_strategy_2024.pdf - Strategy
2. customer_support_handbook.pdf - Support
3. data_privacy_policy.pdf - Legal


In [6]:
# Configure text splitter for optimal chunking
# Documents often require careful chunking to maintain context
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,        # Optimal size for most documents
    chunk_overlap=200,      # Overlap to maintain context between chunks
    length_function=len,    # Use character count for simplicity
    separators=["\n\n", "\n", " ", ""]  # Split on natural boundaries
)

# Process documents into chunks
document_chunks = text_splitter.split_documents(sample_documents)

print("✅ Document chunking completed!")
print(f"Original documents: {len(sample_documents)}")
print(f"Total chunks: {len(document_chunks)}")
print(f"Average chunk size: {sum(len(chunk.page_content) for chunk in document_chunks) / len(document_chunks):.0f} characters")

print("\n📊 Chunking Analysis:")
for i, chunk in enumerate(document_chunks[:3], 1):  # Show first 3 chunks
    print(f"\nChunk {i}:")
    print(f"  Source: {chunk.metadata['source']}")
    print(f"  Size: {len(chunk.page_content)} characters")
    print(f"  Preview: {chunk.page_content[:100]}...")


✅ Document chunking completed!
Original documents: 3
Total chunks: 3
Average chunk size: 733 characters

📊 Chunking Analysis:

Chunk 1:
  Source: enterprise_ai_strategy_2024.pdf
  Size: 673 characters
  Preview: Enterprise AI Strategy 2024
        
        Our company's AI strategy focuses on three key pillars:...

Chunk 2:
  Source: customer_support_handbook.pdf
  Size: 720 characters
  Preview: Customer Support Best Practices
        
        Our customer support team follows these key princip...

Chunk 3:
  Source: data_privacy_policy.pdf
  Size: 807 characters
  Preview: Data Privacy and Security Policy
        
        All employees must comply with the following data ...


## 5. Vector Storage and Embedding Generation

<img src="https://miro.medium.com/1*Djc9EYMV8F3bZ-MxeG09cw.png" alt="Vector Database Architecture" style="max-width: 100%; height: auto;">

Now let's create a vector database to store our document embeddings for efficient semantic search:
Chroma is an AI-native open-source vector database. It comes with everything you need to get started built-in, and runs on your machine.

Reference: https://docs.trychroma.com/docs/overview/getting-started


In [7]:
# Here we are creating a vector store using ChromaDB.
# There are also other popular options for storing vector data, such as Pinecone, Amazon Aurora, and various other specialized vector databases.
vector_store = Chroma.from_documents(
    documents=document_chunks,
    embedding=embeddings,
    persist_directory="./chroma_db",  # Persist data at this location
    collection_name="enterprise_knowledge"
)

print("✅ Vector store created successfully!")
print(f"Total documents: {vector_store._collection.count()}")

# Create retriever from vector store
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4}  # Retrieve top 4 most relevant documents
)

# Define RAG prompt template
rag_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are an intelligent enterprise assistant with access to company knowledge.
    
    Your role is to provide accurate, helpful responses based on the retrieved context.
    Always cite your sources and be specific about which documents you're referencing.
    
    Guidelines:
    - Use only the provided context to answer questions
    - If the context doesn't contain enough information, say so clearly
    - Maintain a professional and appropriate tone
    - Include relevant metadata (department, source) when referencing documents
    - If asked about topics not in the knowledge base, politely redirect to appropriate resources
    
    Context from knowledge base:
    {context}
    
    Question: {question}
    
    Answer:"""),
    ("human", "{question}")
])

# Create RAG chain
def format_docs(docs):
    """Format retrieved documents for the prompt."""
    return "\n\n".join([
        f"Source: {doc.metadata['source']}\n"
        f"Department: {doc.metadata['department']}\n"
        f"Content: {doc.page_content}"
        for doc in docs
    ])

# Create RAG chain that works with string inputs
def create_rag_response(question: str) -> str:
    """Create a RAG response for a given question."""
    # Retrieve relevant documents using the newer invoke method
    docs = retriever.invoke(question)
    
    # Format documents
    context = format_docs(docs)
    
    # Create the prompt
    formatted_prompt = rag_prompt.format(context=context, question=question)
    
    # Generate response
    response = llm.invoke(formatted_prompt)
    
    return response.content

# Create a simple wrapper for the chain
class RAGChain:
    def invoke(self, input_dict):
        question = input_dict["question"]
        return create_rag_response(question)

rag_chain = RAGChain()

print("\n✅ RAG chain created successfully!")

✅ Vector store created successfully!
Total documents: 3

✅ RAG chain created successfully!


## 6. Perform Retrieval and Generation


In [8]:
# Test the RAG system with various queries
# This demonstrates how vanilla RAG works in practice

def perform_rag_query(question: str, description: str):
    """Test a single RAG query and display results."""
    print(f"🔍 {description}")
    print(f"Question: {question}")
    print("-" * 60)
    
    try:
        # Get the response from our RAG chain
        response = rag_chain.invoke({"question": question})
        print(f"Answer: {response}")
        
        # Also show what documents were retrieved
        retrieved_docs = retriever.invoke(question)
        print(f"\n📚 Retrieved {len(retrieved_docs)} relevant documents:")
        for i, doc in enumerate(retrieved_docs, 1):
            print(f"  {i}. {doc.metadata['source']} ({doc.metadata['department']})")
        
    except Exception as e:
        print(f"Error: {e}")
    
    print("\n" + "="*80 + "\n")

# Test various scenarios
print("🧪 Testing Vanilla RAG System")
print("="*80)

# Test 1: Strategy-related query
perform_rag_query(
    "What is our AI strategy for 2024?",
    "Strategy Query - Testing retrieval of strategic information"
)

# Test 2: Support-related query
perform_rag_query(
    "What are our customer support response time requirements?",
    "Support Query - Testing retrieval of operational procedures"
)

# Test 3: Compliance-related query
perform_rag_query(
    "What data classification levels do we use?",
    "Compliance Query - Testing retrieval of policy information"
)

# Test 4: Cross-domain query
perform_rag_query(
    "How much budget is allocated for AI initiatives?",
    "Cross-Domain Query - Testing retrieval across multiple documents"
)

🧪 Testing Vanilla RAG System
🔍 Strategy Query - Testing retrieval of strategic information
Question: What is our AI strategy for 2024?
------------------------------------------------------------
Answer: Our AI strategy for 2024 focuses on three key pillars:

1. **Customer Experience Enhancement** through intelligent automation.
2. **Operational Efficiency** via process optimization.
3. **Data-Driven Decision Making** with advanced analytics.

Key initiatives include:
- Implementing RAG systems for customer support.
- Deploying AI agents for internal knowledge management.
- Establishing data governance frameworks for AI model training.

The budget allocation for this strategy is as follows:
- $2.5 million for AI infrastructure.
- $1.8 million for talent acquisition.
- $1.2 million for compliance and security measures.

(Source: enterprise_ai_strategy_2024.pdf, Department: Strategy)

📚 Retrieved 3 relevant documents:
  1. enterprise_ai_strategy_2024.pdf (Strategy)
  2. data_privacy_poli

## 7. Key Takeaways

### What We've Accomplished

1. **Core RAG Implementation**: Built a complete vanilla RAG system with document processing, vector storage, retrieval and generation.
2. **Advanced Features**: Implemented query analysis and structured retrieval for better precision

### Technical Competencies Gained

- **Document Processing**: Loading, chunking, and preparing documents for RAG
- **Vector Storage**: Creating and managing searchable knowledge bases with ChromaDB
- **Semantic Retrieval**: Implementing similarity search
- **Context-Aware Generation**: Using retrieved information to generate accurate, cited responses

### Enterprise Applications
 
- **FAQ Systems**: Answering frequently asked questions for employees or customers using internal documentation
- **Domain-Specific Retrieval**: Enables efficient retrieval of targeted information from dedicated knowledge sources, which is especially valuable in domains with highly specialized content.
---

**Remember**: Vanilla RAG is the foundation. Master these concepts before moving to more complex agentic patterns!